# Mess waste pipeline on Kaggle

**Before running:** Settings → Accelerator → **GPU T4 x2**, and Settings → **Internet on** (needed to download the vision model).

Add two inputs with **+ Add Input**:
1. the code (this repo uploaded as a Kaggle Dataset, e.g. `messwaste-code`), unless you clone it from GitHub below;
2. your data (a Kaggle Dataset containing `csv/` and `frames/<meal_id>/`). Keep it **private**.

Run the cells top to bottom. Cells marked *optional* can be skipped.

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total --format=csv

## 1. Install

In [ ]:
# torch is preinstalled on Kaggle. Qwen2.5-VL needs a recent transformers.
!pip install -q -U "transformers>=4.51" accelerate pyarrow pyyaml python-pptx
# optional extras:
# !pip install -q anthropic          # Claude API backend
# !pip install -q ultralytics        # person detector for the privacy filter

## 2. Get the code

In [ ]:
import os, sys, glob, shutil
CODE_DIR = "/kaggle/working/mess-waste"

# Option A: code uploaded as a Kaggle Dataset (found automatically)
if not os.path.exists(CODE_DIR):
    hits = glob.glob("/kaggle/input/**/messwaste/cli.py", recursive=True)
    if hits:
        shutil.copytree(os.path.dirname(os.path.dirname(hits[0])), CODE_DIR)

# Option B: private GitHub repo. Store a token in Add-ons -> Secrets as GITHUB_TOKEN, then uncomment:
# from kaggle_secrets import UserSecretsClient
# tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
# !git clone -q https://{tok}@github.com/<your-user>/mess-waste.git {CODE_DIR}

assert os.path.exists(CODE_DIR), "Code not found: add the code dataset or use option B."
os.chdir(CODE_DIR)
sys.path.insert(0, CODE_DIR)
print("code at", CODE_DIR)

## 3. Smoke test on fake data (1 minute, no GPU)
This proves the whole pipeline works before you touch real data.

In [ ]:
!python tests/smoke_test.py

## 4. Point at your real data
Leave `MW_RAW_ROOT` unset to auto-detect the first input folder containing `csv/meals.csv`.

In [ ]:
# os.environ["MW_RAW_ROOT"] = "/kaggle/input/messwaste-data"   # folder with csv/ and frames/
os.environ["MW_OUT_ROOT"] = "/kaggle/working/out"

from messwaste.config import load_config
cfg = load_config()
print("raw :", cfg.paths.raw_root)
print("out :", cfg.paths.out_root)

## 5. Ingest and camera privacy filter

In [ ]:
!python -m messwaste.cli ingest
!python -m messwaste.cli frames

## 6. Vision model (GPU)

Load the model once, preview a few frames, then label everything.
If the preview output is empty, repeated symbols or nonsense, fp16 is overflowing: switch to
`Qwen/Qwen2.5-VL-3B-Instruct` with `float32` (fits across the two T4s), or use the Anthropic backend.

In [ ]:
from messwaste import labeling
cfg = load_config()
# cfg.raw["labeling"]["hf_model_id"] = "Qwen/Qwen2.5-VL-3B-Instruct"; cfg.raw["labeling"]["hf_dtype"] = "float32"
backend = labeling.get_backend(cfg, "hf")          # or "anthropic" (key in Add-ons -> Secrets as ANTHROPIC_API_KEY)

In [ ]:
labeling.preview(cfg, n=6, backend_obj=backend)

In [ ]:
# Resumable: re-running skips frames already labelled by this model.
labels = labeling.run_labeling(cfg, backend_obj=backend)
labels.groupby("meal_id")["frame_id"].nunique()

## 7. Hand-label a validation sample *(once per day)*
Download `out/labelling/labeller.html` from the Output panel. Each teammate labels their own copy and saves `human_labels_<initials>.csv` into your data's `csv/` folder (new dataset version).

In [ ]:
!python -m messwaste.cli human-sample --n 100

## 8. Analyse

In [ ]:
!python -m messwaste.cli agreement
!python -m messwaste.cli analyze

## 9. Forecast the next meals and seal them

Add the upcoming meals to `meals.csv` and `subscriptions.csv` first (menu and subscriber count are known in advance).
Run this **before** the meal starts, then immediately post the printed hash in the hackathon channel and
download the JSON (or commit it). `/kaggle/working` is wiped when the session ends.

In [ ]:
!python -m messwaste.cli forecast

In [ ]:
import json, glob
for p in sorted(glob.glob("/kaggle/working/out/forecasts/sealed/*.json")):
    r = json.load(open(p)); print(r["meal_id"], r["made_at"], r["sha256"])

## 10. After the meal: evaluate, dashboard, datasheet

In [ ]:
!python -m messwaste.cli evaluate
!python -m messwaste.cli report
!python -m messwaste.cli slides        # reports/deck.pptx (make the PDF by opening it in PowerPoint/Slides)
from IPython.display import HTML
HTML(open("/kaggle/working/out/reports/dashboard.html", encoding="utf-8").read())

## 11. Submission bundle *(end of hackathon)*

In [ ]:
!python -m messwaste.cli export
!ls -lh /kaggle/working/out/*.zip

### Keeping results between sessions
Click **Save Version → Quick Save** after labelling. In the next session add this notebook's output as an input:
earlier `plate_labels_model.parquet` files under `/kaggle/input` are picked up automatically, so frames are not re-labelled.